# 09 · Train a model on real data

## What you will build

Until now your network has learned from spirals that you generated yourself. Here it meets real measurements: 569 breast-tissue samples, each described by 30 numbers computed from a microscope image and labelled malignant or benign. You will build the pipeline that turns a dataset into an honest result:

1. `train_val_test_split`: a stratified split into training, validation and test sets;
2. `fit_standardizer` and `apply_standardizer`: feature scaling with statistics from the training rows only;
3. `train_model`: a reproducible training run;
4. `best_epoch`: the epoch chosen on validation data;

and you will finish with a single evaluation on the test set, set against a majority-class baseline.

## Learning objectives

- Give the separate jobs of the training, validation and test sets, and explain why a split should be stratified when the classes are unbalanced.
- Standardise features with statistics from the training rows only, and explain what goes wrong when validation or test rows contribute.
- Make a training run reproducible, choose the epoch on validation data and evaluate the test set once.
- Compare a model with a majority-class baseline, and state what a result from 569 samples can and cannot show.

## Dependencies

Tier 1: NumPy and Matplotlib. One cell, tagged `data`, loads the dataset with scikit-learn's `load_breast_cancer`. The data ship inside the scikit-learn package, so nothing is downloaded, and JupyterLite, the course JupyterHub server and Google Colab all provide it. No other cell uses scikit-learn.

> **Not medical advice.** This notebook teaches a workflow on a small research dataset from the 1990s. Nothing in it can diagnose anyone or help to judge a real test result.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: what you built earlier

The hidden cell below defines the pieces from earlier notebooks under their usual names, so this notebook runs on its own. Open it if you want to see them again.

In [ ]:
# 📦 Recap: the pieces from notebooks 01–04, under their usual names.

def relu(Z):
    """ReLU, element by element: max(0, Z)."""
    return np.maximum(0.0, Z)


def relu_grad(Z):
    """The slope of ReLU: 1.0 where Z > 0, else 0.0 (we take 0 at Z = 0)."""
    return (Z > 0).astype(float)


def sigmoid(Z):
    """Numerically stable logistic function 1 / (1 + exp(-Z)), element by element."""
    return np.exp(-np.logaddexp(0.0, -Z))


def softmax(Z):
    """Row-wise softmax of logits Z (batch, classes). Subtracting each row's maximum avoids overflow."""
    exps = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exps / exps.sum(axis=1, keepdims=True)


def dense_forward(X, W, b):
    """Z = X @ W + b. X: (batch, n_in), W: (n_in, n_out), b: (n_out,) -> Z: (batch, n_out)."""
    return X @ W + b


def init_network(sizes, rng):
    """He-initialised layers for sizes such as [2, 64, 3]: a list of {"W": (n_in, n_out), "b": (n_out,)}."""
    return [
        {"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}
        for n_in, n_out in zip(sizes[:-1], sizes[1:])
    ]


def forward(network, X):
    """ReLU hidden layers and a softmax output.

    Returns (probs, cache): probs has shape (batch, classes), and cache[i] = (A_prev, Z) for layer i.
    """
    cache = []
    A = X
    for index, layer in enumerate(network):
        Z = dense_forward(A, layer["W"], layer["b"])
        cache.append((A, Z))
        A = softmax(Z) if index == len(network) - 1 else relu(Z)
    return A, cache


def predict(network, X):
    """The predicted class index for every row of X: shape (batch,)."""
    probs, _ = forward(network, X)
    return probs.argmax(axis=1)


def accuracy(network, X, y):
    """The fraction of rows of X whose predicted class equals the label in y."""
    return float(np.mean(predict(network, X) == y))


def make_spirals(n_per_class, n_classes, noise, rng):
    """n_classes interleaved spiral arms in 2-D, sorted by class.

    Returns X with shape (n_per_class * n_classes, 2) and integer labels y with
    shape (n_per_class * n_classes,). noise is the spread of the random angle jitter.
    """
    radius = np.linspace(0.05, 1.0, n_per_class)
    X_parts, y_parts = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 6.0 * radius + rng.normal(0.0, noise, n_per_class)
        X_parts.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        y_parts.append(np.full(n_per_class, k))
    return np.concatenate(X_parts), np.concatenate(y_parts)


# From notebook 05: loss functions.

def mse(pred, target):
    """Mean squared error over every entry of two arrays with the same shape."""
    return float(np.mean((pred - target) ** 2))


def binary_cross_entropy(p, y):
    """Mean binary cross-entropy of probabilities p (batch,) against 0/1 labels y (batch,), clipped."""
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def cross_entropy(probs, y):
    """Mean negative log-probability of the true class. probs: (batch, classes), y: (batch,) ints."""
    picked = probs[np.arange(len(y)), y]
    return float(-np.mean(np.log(np.clip(picked, 1e-12, None))))


def cross_entropy_from_logits(Z, y):
    """The same loss computed from logits Z (batch, classes) with the log-sum-exp trick."""
    row_max = Z.max(axis=1, keepdims=True)
    log_sum_exp = row_max[:, 0] + np.log(np.exp(Z - row_max).sum(axis=1))
    return float(np.mean(log_sum_exp - Z[np.arange(len(y)), y]))


# From notebook 06: numerical derivatives and gradient checking.

def numerical_derivative(f, x, h=1e-5):
    """Central-difference estimate of f'(x) for a function of one number."""
    return (f(x + h) - f(x - h)) / (2 * h)


def numerical_gradient(f, array, eps=1e-5):
    """Central-difference gradient of the scalar f(array) with respect to every entry of array.

    Each entry is nudged in place and then restored, so f may also read the array
    through a closure (for example a network's weights). Returns an array shaped like array.
    """
    grad = np.zeros(array.shape)
    for index in np.ndindex(array.shape):
        original = array[index]
        array[index] = original + eps
        loss_plus = f(array)
        array[index] = original - eps
        loss_minus = f(array)
        array[index] = original
        grad[index] = (loss_plus - loss_minus) / (2 * eps)
    return grad


def relative_error(a, b):
    """||a - b|| / (||a|| + ||b||), with a tiny floor on the denominator: 0 when equal, at most 1."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

# From notebook 07: backpropagation.

def softmax_cross_entropy_grad(probs, y):
    """dL/dZ for softmax + mean cross-entropy: (P - Y) / n."""
    return (probs - np.eye(probs.shape[1])[y]) / len(y)


def dense_backward(dZ, A_prev, W):
    """Gradients of a dense layer: (dW, db, dA_prev)."""
    return A_prev.T @ dZ, dZ.sum(axis=0), dZ @ W.T


def backward(network, cache, probs, y):
    """Every layer's gradients, last layer first, returned in layer order."""
    grads = [None] * len(network)
    dZ = softmax_cross_entropy_grad(probs, y)
    for i in reversed(range(len(network))):
        A_prev, Z = cache[i]
        if i < len(network) - 1:
            dZ = dA * relu_grad(Z)
        dW, db, dA = dense_backward(dZ, A_prev, network[i]["W"])
        grads[i] = {"W": dW, "b": db}
    return grads


# From notebook 08: training with gradient descent.

def sgd_step(network, grads, lr):
    """A new network one gradient-descent step downhill: W - lr * dW, b - lr * db."""
    return [{"W": layer["W"] - lr * g["W"], "b": layer["b"] - lr * g["b"]} for layer, g in zip(network, grads)]


def minibatch_indices(n, batch_size, seed):
    """A shuffled split of range(n) into batches of batch_size (the last may be smaller)."""
    order = np.random.default_rng(seed).permutation(n)
    return [order[start:start + batch_size] for start in range(0, n, batch_size)]


def train_epoch(network, X, y, lr, batch_size, seed):
    """One pass over the data in shuffled mini-batches. Returns (network, mean batch loss)."""
    losses = []
    for batch in minibatch_indices(len(X), batch_size, seed):
        probs, cache = forward(network, X[batch])
        losses.append(cross_entropy(probs, y[batch]))
        network = sgd_step(network, backward(network, cache, probs, y[batch]), lr)
    return network, float(np.mean(losses))


def train(network, X, y, X_val, y_val, lr, epochs, batch_size, seed=0):
    """Train for some epochs; returns (network, history of losses and accuracies per epoch)."""
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        network, _ = train_epoch(network, X, y, lr, batch_size, seed + epoch)
        history["train_loss"].append(cross_entropy(forward(network, X)[0], y))
        history["val_loss"].append(cross_entropy(forward(network, X_val)[0], y_val))
        history["train_acc"].append(accuracy(network, X, y))
        history["val_acc"].append(accuracy(network, X_val, y_val))
    return network, history

## 1 · Meet the data

The Wisconsin Diagnostic Breast Cancer data come from fine-needle aspirates: a thin needle draws cells from a breast mass, and the cells are photographed under a microscope. Software traced the outline of every cell nucleus in the image and measured ten properties: radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry and fractal dimension. Each property is summarised in three ways across the nuclei of one sample (the mean, the standard error and the "worst" value, the mean of the three largest), which gives 30 features per sample.

In scikit-learn's copy the label is **0 for malignant and 1 for benign**. Many people expect the opposite, so check the label names whenever you load a dataset.

Shapes: `X_all` is `(569, 30)` and holds float64 measurements; `y_all` is `(569,)` and holds integer labels.

In [ ]:
# The only cell that uses scikit-learn: it loads the copy of the dataset bundled with the package.
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_all = cancer.data.astype(np.float64)    # (569, 30) measurements
y_all = cancer.target.astype(np.int64)    # (569,) labels: 0 = malignant, 1 = benign
feature_names = [str(name) for name in cancer.feature_names]
class_names = [str(name) for name in cancer.target_names]
print(X_all.shape, y_all.shape, class_names)

In [ ]:
counts = np.bincount(y_all)
for label, name in enumerate(class_names):
    print(f"{name:>9}: {counts[label]} samples ({counts[label] / len(y_all):.1%})")
print()
print(f"{'feature':<22}{'smallest':>10}{'largest':>10}")
for column in (0, 3, 4, 7, 23):
    print(f"{feature_names[column]:<22}{X_all[:, column].min():>10.4g}{X_all[:, column].max():>10.4g}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
for label, marker in ((1, "o"), (0, "^")):
    rows = y_all == label
    ax.scatter(X_all[rows, 0], X_all[rows, 7], s=12, marker=marker, alpha=0.6, label=class_names[label])
ax.set_xlabel(feature_names[0])
ax.set_ylabel(feature_names[7])
ax.set_title("two of the 30 features")
ax.legend()
plt.show()

**What to notice.** There are 212 malignant samples (37.3%) and 357 benign ones (62.7%): the classes are unbalanced. The features live on very different scales: `mean area` runs from about 144 to 2501 and `worst area` reaches 4254, while `mean smoothness` stays between 0.05 and 0.16. In the plot the two classes overlap, but larger nuclei with more concave points are mostly malignant, so there is a pattern to learn.

## 2 · Split the data three ways

A model has to be judged on data it never learned from, and choices about the model have to be made without looking at the final exam. So the samples get three jobs:

- the **training** set (60%) fits the weights;
- the **validation** set (20%) makes decisions: which epoch to keep, which learning rate, which architecture;
- the **test** set (20%) is used once, at the very end, to estimate how the chosen model does on new data.

With 37% malignant samples, a purely random split can be unlucky: one test set of 115 samples might hold 35 malignant cases and another 50. A **stratified** split cuts each class separately in the same proportions, so every set keeps roughly the dataset's 37 : 63 mix. For a class with $n_c$ samples, the training set gets $\operatorname{round}(0.6\,n_c)$ of them, the validation set $\operatorname{round}(0.2\,n_c)$ and the test set the rest.

Shapes: `X` is `(n, d)` and `y` is `(n,)`. Six arrays come out, `X_train` `(n_train, d)` and `y_train` `(n_train,)`, and likewise for validation and test, with $n_\text{train} + n_\text{val} + n_\text{test} = n$.

The demo draws five purely random test sets of 115 samples and counts the malignant share in each.

In [ ]:
demo_rng = np.random.default_rng(7)
for trial in range(5):
    rows = demo_rng.permutation(len(y_all))[:115]
    print(f"random test set {trial + 1}: {np.mean(y_all[rows] == 0):.1%} malignant")

### ✍️ Your turn: `train_val_test_split`

Write `train_val_test_split(X, y, fractions, seed)`, returning `X_train, y_train, X_val, y_val, X_test, y_test`.

- `fractions` is a triple such as `(0.6, 0.2, 0.2)`, and `seed` is an integer.
- Make one generator, `rng = np.random.default_rng(seed)`. For each label in `np.unique(y)` (in increasing order), shuffle that class's row indices with `rng.permutation(np.flatnonzero(y == label))`.
- Give the first `int(round(fractions[0] * n_label))` shuffled rows to training, the next `int(round(fractions[1] * n_label))` to validation and the rest to test.
- Join each set's index arrays class by class with `np.concatenate`, then index `X` and `y` with them.

Hints: keep three lists, one per set, and append one index array per class to each. `rows[a:a + b]` is the middle slice.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def train_val_test_split_reference(X, y, fractions, seed):
    rng = np.random.default_rng(seed)
    parts = ([], [], [])
    for label in np.unique(y):
        rows = rng.permutation(np.flatnonzero(y == label))
        n_train = int(round(fractions[0] * len(rows)))
        n_val = int(round(fractions[1] * len(rows)))
        for part, chunk in zip(parts, (rows[:n_train], rows[n_train:n_train + n_val], rows[n_train + n_val:])):
            part.append(chunk)
    train, val, test = (np.concatenate(part) for part in parts)
    return X[train], y[train], X[val], y[val], X[test], y[test]

In [ ]:
def train_val_test_split(X, y, fractions, seed):
    """Stratified split. Returns X_train, y_train, X_val, y_val, X_test, y_test."""
    # ✍️ Your code here (about 10 lines)
    raise NotImplementedError

In [ ]:
toy_X = np.arange(40, dtype=float).reshape(20, 2)
toy_y = np.array([0] * 10 + [1] * 10)
uneven_y = np.array([0] * 7 + [1] * 3)    # edge case: class 1 is too small to reach the test set
three_y = np.array([0, 1, 2] * 8)
train_val_test_split = practice("train_val_test_split", train_val_test_split, train_val_test_split_reference, cases=[
    (toy_X, toy_y, (0.6, 0.2, 0.2), 0),
    (toy_X[:10], uneven_y, (0.6, 0.2, 0.2), 1),
    (np.arange(48, dtype=float).reshape(24, 2), three_y, (0.5, 0.25, 0.25), 2),
    (X_all, y_all, (0.6, 0.2, 0.2), 3),
])

**Expected behaviour.** The random demo gives malignant shares between about 36% and 46%, against 37.3% in the whole dataset. Your function passes the four checks, including the uneven case: its minority class has only three rows, so two go to training, one to validation and none to test. Stratification cannot conjure up samples a class does not have.

In [ ]:
SPLIT_SEED = 0
X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(X_all, y_all, (0.6, 0.2, 0.2), SPLIT_SEED)
for name, labels in (("training", y_train), ("validation", y_val), ("test", y_test)):
    print(f"{name:>10}: {len(labels):3d} samples, {np.mean(labels == 0):.1%} malignant")

**Expected behaviour.** 341 training, 113 validation and 115 test samples (341 + 113 + 115 = 569), each with 37.2% to 37.4% malignant samples. Every sample is in exactly one set.

## 3 · A baseline to beat

Before training anything, ask what the laziest sensible rule would score. The **majority-class baseline** predicts the most common training label for every sample. Here that is "benign", so it is right about 63% of the time, and it never finds a single malignant sample. A model worth having must beat it clearly, and on the samples that matter.

In [ ]:
majority = int(np.bincount(y_train).argmax())
baseline_val = float(np.mean(y_val == majority))
print(f"majority training class: {class_names[majority]}")
print(f"baseline validation accuracy: {baseline_val:.3f}")

## 4 · Standardise the features

The first layer computes $Z = XW + b$, adding up `area × weight` and `smoothness × weight`. With weights of similar size, a feature in the thousands drowns one below 0.2. The gradients are just as lopsided, because $\partial L/\partial W = X^\top\,\partial L/\partial Z$ scales with the inputs, so no single learning rate suits every weight: a rate small enough for the area weights barely moves the smoothness weights.

**Standardisation** rescales every column $j$ to mean 0 and standard deviation 1:

$$\mu_j = \frac{1}{n}\sum_{i=1}^{n} x_{ij}, \qquad \sigma_j = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (x_{ij} - \mu_j)^2}, \qquad z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}.$$

Shapes: `X_train` is `(n_train, 30)`, so `mean` and `std` are `(30,)`. Broadcasting then subtracts and divides row by row for any `X` of shape `(n, 30)`.

**Fit on the training rows only.** The scaler is part of the model: its 60 numbers are learned from data just like the weights. The validation and test rows stand in for future samples that you will not have when you fit the model, so their statistics must not shape it. On this dataset the leak would be small (the means would move by less than a tenth of a standard deviation), but the habit matters: the same mistake with steps that use the labels, such as choosing features or oversampling before the split, can inflate a score dramatically.

A constant column has $\sigma_j = 0$, and dividing by zero gives `inf` or `nan`. Replacing a zero standard deviation by 1 leaves such a column at 0 after centring.

In [ ]:
area = X_train[:, 3]
z = (area - area.mean()) / area.std()
print(f"mean area: average {area.mean():.1f}, standard deviation {area.std():.1f}")
print(f"standardised: average {z.mean():.1e}, standard deviation {z.std():.3f}")

### ✍️ Your turn: `fit_standardizer`

Write `fit_standardizer(X_train)`, returning `(mean, std)`: the mean and the standard deviation of every column (`axis=0`; NumPy's default `ddof=0` is the formula above), each with shape `(d,)`, with every zero in `std` replaced by `1.0`.

Hints: `X_train.mean(axis=0)`, `X_train.std(axis=0)` and `np.where(std == 0, 1.0, std)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def fit_standardizer_reference(X_train):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    return mean, np.where(std == 0, 1.0, std)

In [ ]:
def fit_standardizer(X_train):
    """Column means and standard deviations of X_train (n, d), each (d,); zero stds become 1.0."""
    # ✍️ Your code here (about 3 lines)
    raise NotImplementedError

In [ ]:
fit_standardizer = practice("fit_standardizer", fit_standardizer, fit_standardizer_reference, cases=[
    (np.array([[1.0, 10.0], [3.0, 30.0], [5.0, 20.0]]),),
    (np.array([[2.0, 5.0], [2.0, 7.0]]),),    # edge case: the first column is constant
    (X_train,),
])

**Expected behaviour.** The demo turns `mean area` (average about 642, standard deviation about 338 in the training rows) into a column with average 0 and standard deviation 1. For the first check, the means are `[3, 20]` and the standard deviations `[1.633, 8.165]`; in the second, the constant column gets a standard deviation of 1, not 0.

Fitting and applying are separate steps on purpose. You **fit** once, on the training rows, and then **apply** the same 60 numbers to every set, and later to every new sample the model meets. If each set were scaled with its own statistics, the same measurement would turn into different inputs depending on which set it happened to be in.

In [ ]:
demo_mean, demo_std = np.array([10.0, 0.1]), np.array([2.0, 0.02])
print((np.array([[12.0, 0.08], [10.0, 0.1]]) - demo_mean) / demo_std)

### ✍️ Your turn: `apply_standardizer`

Write `apply_standardizer(X, mean, std)`, returning `(X - mean) / std`, an array with the same shape as `X`, for any number of rows. It must not change `X` in place.

Hint: one line; broadcasting does the rest.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def apply_standardizer_reference(X, mean, std):
    return (X - mean) / std

In [ ]:
def apply_standardizer(X, mean, std):
    """Scale X (n, d) with training statistics mean (d,) and std (d,)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
check_mean, check_std = np.array([3.0, 20.0]), np.array([2.0, 10.0])
apply_standardizer = practice("apply_standardizer", apply_standardizer, apply_standardizer_reference, cases=[
    (np.array([[1.0, 10.0], [3.0, 30.0], [5.0, 20.0]]), check_mean, check_std),
    (np.array([[3.0, 20.0]]), check_mean, check_std),    # edge case: a single row
    (X_val, X_train.mean(axis=0), X_train.std(axis=0)),
])

**Expected behaviour.** The demo prints `[[1, -1], [0, 0]]`: 12 is one standard deviation above 10, and 0.08 is one below 0.1. The single-row case returns `[[0, 0]]`, still with two dimensions.

In [ ]:
mean, std = fit_standardizer(X_train)
X_train_s = apply_standardizer(X_train, mean, std)
X_val_s = apply_standardizer(X_val, mean, std)
X_test_s = apply_standardizer(X_test, mean, std)
print(f"training columns: largest |mean| {np.abs(X_train_s.mean(axis=0)).max():.1e}, "
      f"standard deviations {X_train_s.std(axis=0).min():.3f} to {X_train_s.std(axis=0).max():.3f}")
print("validation column means, first five:", X_val_s.mean(axis=0)[:5].round(3))

**Expected behaviour.** The training columns have means of about $10^{-14}$ or smaller (zero up to rounding) and standard deviations of 1. The validation means are near 0 but not equal to it, for example about $0.19$ or $-0.16$. That is correct: the validation rows are scaled with the training statistics, exactly as future samples will be.

## 5 · A reproducible training run

A training run is a recipe: an architecture (the layer sizes), hyperparameters (the learning rate, the number of epochs and the batch size) and seeds for its two random ingredients, the starting weights and the order of the mini-batches. Fix all of them and training is **reproducible**: running it twice gives the same weights, bit for bit. That lets you replay a run to a chosen epoch in section 6, and it lets anyone check your result.

The network is `[30, 16, 2]`: 30 inputs, 16 ReLU hidden units and 2 softmax outputs (malignant and benign), with $30 \times 16 + 16 + 16 \times 2 + 2 = 530$ parameters.

In [ ]:
first = init_network([30, 16, 2], np.random.default_rng(0))
second = init_network([30, 16, 2], np.random.default_rng(0))
print("same seed, same starting weights:", all(np.array_equal(a["W"], b["W"]) for a, b in zip(first, second)))
print("parameters:", sum(layer["W"].size + layer["b"].size for layer in first))

### ✍️ Your turn: `train_model`

Write `train_model(X_train, y_train, X_val, y_val, sizes, lr, epochs, batch_size, seed)`, returning `(network, history)`:

1. `network = init_network(sizes, np.random.default_rng(seed))`;
2. return what `train(network, X_train, y_train, X_val, y_val, lr, epochs, batch_size, seed)` returns: the trained network and its history, with one entry per epoch in `train_loss`, `val_loss`, `train_acc` and `val_acc`.

The same `seed` fixes both random ingredients: `init_network` draws the weights from it, and `train` shuffles epoch `k` with seed `seed + k`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def train_model_reference(X_train, y_train, X_val, y_val, sizes, lr, epochs, batch_size, seed):
    network = init_network(sizes, np.random.default_rng(seed))
    return train(network, X_train, y_train, X_val, y_val, lr, epochs, batch_size, seed)

In [ ]:
def train_model(X_train, y_train, X_val, y_val, sizes, lr, epochs, batch_size, seed):
    """A reproducible run: He-initialise from seed, train, return (network, history)."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
tiny_X = np.random.default_rng(5).normal(size=(12, 3))
tiny_y = np.array([0, 1] * 6)
train_model = practice("train_model", train_model, train_model_reference, cases=[
    (tiny_X, tiny_y, tiny_X[:4], tiny_y[:4], [3, 4, 2], 0.1, 3, 5, 0),
    (tiny_X, tiny_y, tiny_X[:4], tiny_y[:4], [3, 2], 0.5, 2, 12, 7),    # edge case: no hidden layer, one batch
])

**Expected behaviour.** The demo prints `True` and `530`. Your `train_model` returns the same network and the same history as the reference: identical seeds give identical runs. The second check case has no hidden layer at all, which makes the network a softmax regression; the same code handles it.

In [ ]:
SIZES = [30, 16, 2]
LEARNING_RATE = 0.05
EPOCHS = 100
BATCH_SIZE = 32
SEED = 0

network, history = train_model(X_train_s, y_train, X_val_s, y_val, SIZES, LEARNING_RATE, EPOCHS, BATCH_SIZE, SEED)
print(f"after epoch 1:   training loss {history['train_loss'][0]:.3f}, validation loss {history['val_loss'][0]:.3f}")
print(f"after epoch {EPOCHS}: training loss {history['train_loss'][-1]:.3f}, validation loss {history['val_loss'][-1]:.3f}")
print(f"final validation accuracy {history['val_acc'][-1]:.3f} (baseline {baseline_val:.3f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(history["train_loss"], label="training")
axes[0].plot(history["val_loss"], label="validation")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy")
axes[0].legend()
axes[1].plot(history["train_acc"], label="training")
axes[1].plot(history["val_acc"], label="validation")
axes[1].axhline(baseline_val, color="grey", linestyle=":", label="majority baseline")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()

**Expected behaviour.** The training loss falls from about 0.33 after the first epoch to about 0.02 after the hundredth. The validation loss falls quickly to about 0.175 near epoch 18 and then creeps back up to about 0.20, while the validation accuracy settles near 0.94, far above the baseline's 0.63. The rising validation loss is the first sign of overfitting: the network keeps sharpening its fit to the 341 training samples, and the extra confidence does not carry over to new ones.

## 6 · Choose the epoch on validation data

The validation curve tells you when to stop. The epoch to keep is the one with the **lowest validation loss**. Why loss rather than accuracy? With 113 validation samples, accuracy moves in steps of $1/113 \approx 0.9$ percentage points, so many epochs tie, whereas the loss changes smoothly and also rewards being confidently right.

Epochs are counted from 0 here, like positions in the history lists: `history["val_loss"][k]` was measured after `k + 1` epochs of training.

In [ ]:
losses = [0.9, 0.5, 0.42, 0.45, 0.6]
print("lowest validation loss", min(losses), "at index", losses.index(min(losses)))

### ✍️ Your turn: `best_epoch`

Write `best_epoch(history)`, returning the index (a Python `int`) of the smallest value in `history["val_loss"]`; when several epochs tie, the earliest.

Hint: `np.argmin` returns the first position of the minimum; wrap it in `int(...)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def best_epoch_reference(history):
    return int(np.argmin(history["val_loss"]))

In [ ]:
def best_epoch(history):
    """Index of the epoch with the lowest validation loss (the earliest on ties)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
best_epoch = practice("best_epoch", best_epoch, best_epoch_reference, cases=[
    ({"val_loss": [0.9, 0.5, 0.7]},),
    ({"val_loss": [0.4, 0.3, 0.3, 0.5]},),    # edge case: a tie, so the earlier epoch wins
    ({"val_loss": [1.0]},),
    (history,),
])

**Expected behaviour.** The demo finds 0.42 at index 2. Your function returns 1, 1 (the tie goes to the earlier epoch), 0, and the best epoch of the real run.

To get the network from that epoch, replay the run: with the same seeds, the first `chosen + 1` epochs of a new run are exactly the same computation as in the long run. Notebook 10 shows the other way: keep a copy of the best weights while training.

In [ ]:
chosen = best_epoch(history)
final_network, final_history = train_model(X_train_s, y_train, X_val_s, y_val, SIZES, LEARNING_RATE,
                                           chosen + 1, BATCH_SIZE, SEED)
print(f"lowest validation loss in the long run: {history['val_loss'][chosen]:.4f} at epoch index {chosen}")
print(f"replayed run of {chosen + 1} epochs ends at:  {final_history['val_loss'][-1]:.4f}")

**Expected behaviour.** Both lines show the same validation loss, about 0.175, at epoch index 18. The replay reproduces the chosen network exactly because nothing random is left to chance.

## 7 · Evaluate on the test set, once

Every decision is now made: the split, the scaler, the architecture, the learning rate and the epoch. Only now do we look at the test set, and only once. If you looked at the test score and then changed something, the test set would become another validation set, and you would need fresh data for an honest estimate.

In [ ]:
test_probs, _ = forward(final_network, X_test_s)
test_pred = test_probs.argmax(axis=1)
test_accuracy = accuracy(final_network, X_test_s, y_test)
baseline_test = float(np.mean(y_test == majority))
missed = int(np.sum((y_test == 0) & (test_pred == 1)))
false_alarms = int(np.sum((y_test == 1) & (test_pred == 0)))
print(f"test accuracy {test_accuracy:.3f} on {len(y_test)} samples (majority baseline {baseline_test:.3f})")
print(f"test loss {cross_entropy(test_probs, y_test):.3f}")
print(f"malignant samples called benign: {missed} of {int(np.sum(y_test == 0))}")
print(f"benign samples called malignant: {false_alarms} of {int(np.sum(y_test == 1))}")

**Expected behaviour.** A test accuracy of about 0.957 against a baseline of 0.626: 110 of 115 samples right. One of the 43 malignant samples is called benign and four of the 72 benign ones are called malignant.

**How precise is that number?** One sample is worth $1/115 \approx 0.9$ percentage points. A rough 95% interval for an accuracy $p$ measured on $n$ samples is $p \pm 1.96\sqrt{p(1-p)/n}$, here about $0.957 \pm 0.037$. The two kinds of mistake also differ in cost: a missed malignant sample is far more serious than a false alarm, yet accuracy counts them the same. Notebook 10 builds precision and recall for exactly this.

## 8 · 🧪 Experiments

### 🧪 What if you skip standardisation?

Train the same network on the raw measurements and on the standardised ones, with the same learning rate. Change `experiment_lr` and run the cell again.

In [ ]:
experiment_lr = 0.05  # 🧪 try 0.001 and 0.00001, then run this cell again
for label, (train_inputs, val_inputs) in (("raw", (X_train, X_val)), ("standardised", (X_train_s, X_val_s))):
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        _, h = train_model(train_inputs, y_train, val_inputs, y_val, SIZES, experiment_lr, EPOCHS, BATCH_SIZE, SEED)
    print(f"{label:>12}: training loss after epoch 1 {h['train_loss'][0]:7.3f}, after epoch {EPOCHS} "
          f"{h['train_loss'][-1]:.3f}; best validation accuracy {max(h['val_acc']):.3f}")

**What to notice.** At 0.05 the raw run is stuck at the baseline: its first steps are so large that the network collapses to calling every sample benign, and its best validation accuracy is 0.628, while the standardised run reaches about 0.94. Lower the rate and the picture flips: at 0.001 the raw run reaches about 0.94 while the standardised one, now stepping too timidly, reaches about 0.89, and at 0.00001 the standardised run hardly moves at all. Each scale of input needs its own learning rate, and these raw features need one about 50 times smaller, set by the units the measurements happen to be in. Standardising puts every dataset on the same footing, so a learning rate that works on one is a sensible starting point for the next. That is why it is the default first step.

### 🧪 How much does the split matter?

Repeat the whole procedure (split, scale, train, choose the epoch, test once) with different split seeds. Each seed draws a different test set of 115 samples.

In [ ]:
split_seeds = [0, 1, 2, 3, 4, 5]  # 🧪 try other seeds
scores = []
for split_seed in split_seeds:
    X_tr, y_tr, X_va, y_va, X_te, y_te = train_val_test_split(X_all, y_all, (0.6, 0.2, 0.2), split_seed)
    m, s = fit_standardizer(X_tr)
    X_tr, X_va, X_te = (apply_standardizer(part, m, s) for part in (X_tr, X_va, X_te))
    _, h = train_model(X_tr, y_tr, X_va, y_va, SIZES, LEARNING_RATE, EPOCHS, BATCH_SIZE, SEED)
    k = best_epoch(h)
    kept, _ = train_model(X_tr, y_tr, X_va, y_va, SIZES, LEARNING_RATE, k + 1, BATCH_SIZE, SEED)
    scores.append(accuracy(kept, X_te, y_te))
    print(f"split seed {split_seed}: kept epoch index {k:3d}, test accuracy {scores[-1]:.3f}")
print(f"range of test accuracies: {min(scores):.3f} to {max(scores):.3f}")

**What to notice.** The test accuracies range from about 0.957 to 0.983 across the six splits, and the kept epoch moves a lot (from 17 to 99). A difference of one or two points between two models measured on one split of this size is within this noise. Studying the spread is legitimate because no decision is taken from it; picking the luckiest seed and reporting its score would not be.

## 9 · Put it together

`run_pipeline` chains your functions into the whole protocol. Read it from top to bottom: every line is one of the steps above.

In [ ]:
def run_pipeline(X, y, split_seed, seed):
    """Split, standardise, train, choose the epoch on validation data, then test once."""
    X_tr, y_tr, X_va, y_va, X_te, y_te = train_val_test_split(X, y, (0.6, 0.2, 0.2), split_seed)
    mean_tr, std_tr = fit_standardizer(X_tr)
    X_tr, X_va, X_te = (apply_standardizer(part, mean_tr, std_tr) for part in (X_tr, X_va, X_te))
    _, full_history = train_model(X_tr, y_tr, X_va, y_va, SIZES, LEARNING_RATE, EPOCHS, BATCH_SIZE, seed)
    kept_epoch = best_epoch(full_history)
    kept, _ = train_model(X_tr, y_tr, X_va, y_va, SIZES, LEARNING_RATE, kept_epoch + 1, BATCH_SIZE, seed)
    return {
        "set_sizes": (len(y_tr), len(y_va), len(y_te)),
        "scaler_mean": mean_tr,
        "history": full_history,
        "kept_epoch": kept_epoch,
        "kept_val_loss": cross_entropy(forward(kept, X_va)[0], y_va),
        "test_accuracy": accuracy(kept, X_te, y_te),
        "baseline_accuracy": float(np.mean(y_te == np.bincount(y_tr).argmax())),
    }


result = run_pipeline(X_all, y_all, SPLIT_SEED, SEED)
print(f"sets {result['set_sizes']}, kept epoch index {result['kept_epoch']}")
print(f"test accuracy {result['test_accuracy']:.3f} against a baseline of {result['baseline_accuracy']:.3f}")

assert sum(result["set_sizes"]) == len(y_all), "every sample must be in exactly one set"
assert np.allclose(result["scaler_mean"], X_train.mean(axis=0)), "the scaler must be fitted on the training rows only"
assert result["history"]["train_loss"][-1] < 0.5 * result["history"]["train_loss"][0], "training should more than halve the loss"
assert np.isclose(result["kept_val_loss"], min(result["history"]["val_loss"])), "the kept network must be the best validation epoch"
assert result["test_accuracy"] >= 0.90, f"expected at least 0.90 test accuracy, got {result['test_accuracy']:.3f}"
assert result["test_accuracy"] >= result["baseline_accuracy"] + 0.20, "the network should clearly beat the baseline"
print("✅ Split, scaled on training rows, trained, chose the epoch on validation data, tested once.")

## 10 · What this result does and does not show

- **A small sample.** With 115 test samples, one more mistake moves the accuracy by 0.9 points, and a rough 95% interval spans about ±4 points. The split experiment showed a spread of about 2.6 points from the split alone.
- **One dataset, one clinic, one period.** The samples come from patients of one hospital (the University of Wisconsin) in the early 1990s, imaged and measured with one procedure. Other clinics, microscopes, software or patient groups may differ, and this notebook has no data to say how much.
- **An artificial class mix.** 37% of these samples are malignant because of how the dataset was assembled. Among the people tested in practice the share differs, and that changes how often a "malignant" prediction is right (exercise 3).
- **Accuracy hides the cost of mistakes.** A missed malignant sample and a false alarm count the same here; in reality they do not.
- **No external validation, no clinical study, no check of the probabilities' calibration.**
- **Not medical advice.** This notebook teaches a workflow. It must not be used to make or inform any decision about anyone's health.

## Exercises

### Exercise 1 · ✍️ An interval for an accuracy

Write `accuracy_interval(acc, n, z=1.96)`, returning `(low, high)` $= p \pm z\sqrt{p(1-p)/n}$ with $p$ = `acc`, each clipped to the range $[0, 1]$ (use `max` and `min`, or `np.clip`). Return two Python floats. Then use it on your test accuracy: does the interval include 0.90? Does it include the baseline?

In [ ]:
def accuracy_interval_reference(acc, n, z=1.96):
    half_width = z * np.sqrt(acc * (1 - acc) / n)
    return float(max(acc - half_width, 0.0)), float(min(acc + half_width, 1.0))

In [ ]:
def accuracy_interval(acc, n, z=1.96):
    """Normal-approximation interval (low, high) for an accuracy measured on n samples."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
accuracy_interval = practice("accuracy_interval", accuracy_interval, accuracy_interval_reference, cases=[
    (0.957, 115), (0.5, 100), (1.0, 50), (0.99, 20, 2.58),
])
print("test accuracy interval:", accuracy_interval(test_accuracy, len(y_test)))

### Exercise 2 · ✍️ Move the decision threshold

The network calls a sample malignant when $P(\text{malignant}) > 0.5$. Write `predict_with_threshold(network, X, threshold)`, returning an integer array of labels with `0` (malignant) wherever the network's probability for class 0 is **at least** `threshold`, and `1` elsewhere. Then count, on the **validation** set, how many malignant samples are missed and how many false alarms you get for thresholds 0.5, 0.3 and 0.1. Which threshold would you choose, and why must the choice be made on validation data?

In [ ]:
def predict_with_threshold_reference(network, X, threshold):
    probs, _ = forward(network, X)
    return np.where(probs[:, 0] >= threshold, 0, 1)

In [ ]:
def predict_with_threshold(network, X, threshold):
    """Labels: 0 (malignant) where P(class 0) >= threshold, else 1 (benign)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
predict_with_threshold = practice("predict_with_threshold", predict_with_threshold, predict_with_threshold_reference, cases=[
    (final_network, X_val_s[:8], 0.5), (final_network, X_val_s, 0.1), (final_network, X_val_s, 1.0),
])
for threshold in (0.5, 0.3, 0.1):
    labels = predict_with_threshold(final_network, X_val_s, threshold)
    print(f"threshold {threshold}: missed malignant {int(np.sum((y_val == 0) & (labels == 1)))}, "
          f"false alarms {int(np.sum((y_val == 1) & (labels == 0)))}")

### Exercise 3 · Open question: the same model, a different population

Suppose that on a population where 37% of samples are malignant, the model finds 97% of the malignant samples (its sensitivity) and correctly clears 95% of the benign ones (its specificity). In a population where only 2% of samples are malignant, what fraction of "malignant" predictions would be right? Use Bayes' rule: $P(\text{malignant} \mid \text{flagged}) = \dfrac{0.97 \times 0.02}{0.97 \times 0.02 + 0.05 \times 0.98}$. What does this say about reporting accuracy alone?

### Exercise 4 · Open question: measure the leak

Fit the standardiser on `X_all` instead of `X_train`, rerun sections 5 to 7, and compare the test accuracy. The difference is probably tiny. Explain why the procedure is still wrong, and describe a preprocessing step whose leak would not be tiny.

## Common errors and debugging

- **Validation accuracy stuck at about 0.63.** The network predicts "benign" for everyone. The usual causes are unscaled inputs or a learning rate that is far too large.
- **The scaler fitted on all the data.** Fit it on `X_train` only, then apply it to all three sets.
- **Validation or test rows left unscaled.** The network then sees inputs on a completely different scale, and its accuracy collapses towards the baseline.
- **The labels the wrong way round.** In this dataset, 0 means malignant and 1 benign.
- **Choosing anything on the test set.** The epoch, the learning rate and the threshold are all choices; make them on validation data.
- **`inf` or `nan` after scaling.** A column has a standard deviation of 0; replace it by 1.
- **Off by one when replaying.** Epoch index `k` needs `k + 1` epochs of training.
- **`X.mean()` without `axis=0`.** That is one number for the whole table instead of one per column.

## Summary

An honest result needs a protocol. You split the data into training, validation and test sets, stratified by class; fitted the scaler on the training rows only and applied it everywhere; made training reproducible with fixed seeds; chose the epoch with the lowest validation loss; and evaluated the test set once, against a majority-class baseline. Your network reached about 96% test accuracy against a baseline of 63%, on a test set small enough that the uncertainty is several points wide.

## Next

**10 · Improve model performance**: diagnose underfitting and overfitting from learning curves, tune the learning rate, batch size and initialisation, add L2 regularisation and early stopping, and measure precision, recall and F1.